In [4]:
import numpy as np

# Jeu de données
# Chaque élément contient :
# (phrase française, traduction anglaise)
data = [
    ("bonjour", "hello"),
    ("merci", "thanks"),
    ("oui", "yes"),
    ("non", "no"),
    ("bonne nuit", "good night"),
    ("je mange", "i eat"),
    ("je cours", "i run"),
    ("je dors", "i sleep"),
    ("chat noir", "black cat"),
    ("chien blanc", "white dog")
]

# On récupère tous les mots français 
# set permet d'éviter les doublons
fr_words = set()

for fr, en in data:
    for word in fr.split():
        fr_words.add(word)
# On trie les mots pour avoir toujours le même ordre
fr_words = sorted(list(fr_words))

# tous les mots anglais possibles
# ici chaque réponse complète est une possibilité
en_words = sorted(list(set(en for fr, en in data)))

fr_word_to_index = {word: i for i, word in enumerate(fr_words)}
en_word_to_index = {word: i for i, word in enumerate(en_words)}

print("Nombre de mots français :", len(fr_words))
print("Nombre de mots anglais :", len(en_words))


# transformer un mot en vecteur one-hot
def one_hot(index, size):
    x = np.zeros((size, 1))
    x[index] = 1
    return x


# taille de la mémoire du GRU
hidden_size = 8

input_size = len(fr_words)
output_size = len(en_words)

np.random.seed(42)

# paramètres de la porte reset
Wr = np.random.randn(hidden_size, input_size + hidden_size) * 0.1
br = np.zeros((hidden_size, 1))

# paramètres de la porte update
Wz = np.random.randn(hidden_size, input_size + hidden_size) * 0.1
bz = np.zeros((hidden_size, 1))

# paramètres de la nouvelle mémoire
Wh = np.random.randn(hidden_size, input_size + hidden_size) * 0.1
bh = np.zeros((hidden_size, 1))

# couche de sortie
Wy = np.random.randn(output_size, hidden_size) * 0.1
by = np.zeros((output_size, 1))

# sigmoid donne des valeurs entre 0 et 1
def sigmoid(x):
    return 1 / (1 + np.exp(-x))

# transforme les scores en probabilités
def softmax(x):
    x = x - np.max(x)
    exp_x = np.exp(x)
    return exp_x / np.sum(exp_x)


def gru_cell(x_t, h_prev):

    # on met le mot actuel et l'ancienne mémoire ensemble
    combined = np.vstack((x_t, h_prev))

    # porte de remise à zéro
    # elle décide quelle partie de l'ancienne mémoire utiliser
    r_t = sigmoid(Wr @ combined + br)

    # on applique la porte reset à l'ancienne mémoire
    reset_memory = r_t * h_prev

    # nouvelle combinaison pour calculer la nouvelle mémoire
    combined_candidate = np.vstack((x_t, reset_memory))

    # nouvelle mémoire proposée
    h_candidate = np.tanh(Wh @ combined_candidate + bh)

    # porte de mise à jour
    # elle décide combien garder de l'ancienne et de la nouvelle mémoire
    z_t = sigmoid(Wz @ combined + bz)

    # nouvelle mémoire finale
    h_t = (1 - z_t) * h_prev + z_t * h_candidate

    return h_t, r_t, z_t


# forward : le GRU lit la phrase mot par mot
def forward(sentence):

    h = np.zeros((hidden_size, 1))

    states = []

    for word in sentence.split():

        x = one_hot(fr_word_to_index[word], input_size)

        h, r, z = gru_cell(x, h)

        states.append((h, r, z))

    # couche de sortie
    y = softmax(Wy @ h + by)

    return y, states


# fonction de perte
def cross_entropy(y, target):
    return -np.log(y[target, 0] + 1e-8)


# entraînement 
learning_rate = 0.1
epochs = 1000

for epoch in range(epochs):

    total_loss = 0
    for fr, en in data:
        # forward
        y, states = forward(fr)

        # mot anglais attendu
        target = en_words.index(en)

        # calcul de l'erreur
        loss = cross_entropy(y, target)
        total_loss += loss

        # gradient de la sortie
        dy = y.copy()
        dy[target] -= 1

        # récupérer le dernier h
        h_last = states[-1][0]

        # mise à jour de la couche de sortie
        dWy = dy @ h_last.T
        dby = dy

        Wy -= learning_rate * dWy
        by -= learning_rate * dby

    if epoch % 100 == 0:
        print("Epoch :", epoch, "Loss :", round(total_loss, 4))


# prédiction
def predict(sentence):
    y, states = forward(sentence)
    index = np.argmax(y)
    return en_words[index]

print("\nRésultats du GRU :\n")
for fr, en in data:

    prediction = predict(fr)

    print(fr, "→", prediction, "(attendu :", en + ")")


# regarder les portes pour une phrase
print("\n\nExemple avec les portes du GRU :")

sentence = "chat noir"

y, states = forward(sentence)

for i, word in enumerate(sentence.split()):

    h, r, z = states[i]

    print("\nMot :", word)

    print("Mémoire h_t :")
    print(h)

    print("Porte de remise à zéro r_t :")
    print(r)

    print("Porte de mise à jour z_t :")
    print(z)

print("\nPrédiction :", predict(sentence))

Nombre de mots français : 14
Nombre de mots anglais : 10
Epoch : 0 Loss : 23.5493
Epoch : 100 Loss : 21.3912
Epoch : 200 Loss : 19.4654
Epoch : 300 Loss : 17.7631
Epoch : 400 Loss : 16.2658
Epoch : 500 Loss : 14.9516
Epoch : 600 Loss : 13.798
Epoch : 700 Loss : 12.7839
Epoch : 800 Loss : 11.89
Epoch : 900 Loss : 11.0993

Résultats du GRU :

bonjour → hello (attendu : hello)
merci → thanks (attendu : thanks)
oui → yes (attendu : yes)
non → no (attendu : no)
bonne nuit → good night (attendu : good night)
je mange → i eat (attendu : i eat)
je cours → i run (attendu : i run)
je dors → i sleep (attendu : i sleep)
chat noir → black cat (attendu : black cat)
chien blanc → white dog (attendu : white dog)


Exemple avec les portes du GRU :

Mot : chat
Mémoire h_t :
[[-0.05671249]
 [ 0.00942302]
 [ 0.0566221 ]
 [ 0.0854237 ]
 [-0.0041378 ]
 [-0.01070286]
 [-0.05682397]
 [-0.04004919]]
Porte de remise à zéro r_t :
[[0.53800232]
 [0.50277304]
 [0.52640347]
 [0.4838776 ]
 [0.52419721]
 [0.56127156]